# Mock A · Classify acoustic panels

Two laboratories tested acoustic panels and exported their columns in different orders. Predict acceptance from measurements. You will be graded on column alignment, boundary cases and what you can say about uncertain predictions.

**Practice conditions** · 180 minutes · 30 practice points. Start with a fresh kernel. Run the supplied setup, then implement each task in its answer cell. Checks run after you fill the preceding cell. Grade the printed examples and plots yourself. Use array operations across records. Loops over batches, epochs, a few printed records and plot annotations are fine.

All data are generated locally. Use Python, NumPy, PyTorch and Matplotlib. The worked notebook is in `solutions/` for review after your attempt.

In [ ]:
import copy
import csv
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
torch.manual_seed(41)

In [ ]:
# Supplied fixture. Treat the resulting CSV strings as the two lab exports.
def lab_export(n, order, seed):
    rng = np.random.default_rng(seed)
    fields = np.array(["density", "thickness", "absorption", "porosity", "rating_a", "rating_b", "rating_c"])
    x = rng.normal(size=(n, 4))
    signal = 0.9*x[:, 0] - 0.8*x[:, 1] + 0.6*x[:, 2]*x[:, 3]
    votes = np.clip(np.rint(5.3 + signal[:, None] + rng.normal(size=(n, 3))*1.5), 1, 9)
    table = np.column_stack((x, votes))
    text = StringIO()
    np.savetxt(text, table[:, order], delimiter=";", header=";".join(fields[order]),
               comments="", fmt="%.4f")
    return text.getvalue()

file_a = lab_export(947, [0, 1, 2, 3, 4, 5, 6], 31)
file_b = lab_export(681, [6, 3, 1, 4, 0, 5, 2], 32)
print(file_a.splitlines()[:2], file_b.splitlines()[:2], sep="\n")

## A · Parse the exports · 1 point

Write `read_export(text)` to return an array of header names and a 2-D float32 array of data in file order. Parse both strings into `names_a, a_rows` and `names_b, b_rows`.

In [ ]:
def read_export(text):
    header = np.array(text.splitlines()[0].split(";"))
    rows = np.loadtxt(StringIO(text), delimiter=";", skiprows=1, dtype=np.float32, ndmin=2)
    return header, rows

names_a, a_rows = read_export(file_a)
names_b, b_rows = read_export(file_b)

In [ ]:
# Check your work
assert a_rows.shape == (947, 7) and b_rows.shape == (681, 7)
assert a_rows.dtype == b_rows.dtype == np.float32
assert list(names_b) == file_b.splitlines()[0].split(";")
assert np.allclose(b_rows[0], np.fromstring(file_b.splitlines()[1], sep=";"))
print("Check passed.")

## B · Align by header and merge · 6 points

Implement `merge(left, left_names, right, right_names)`. Put all left rows first. Assume each name occurs exactly once on either side. Earn 2 points for a correct merge on these files, +1 for arbitrary permutations, +1 for vectorized column reordering, +2 for satisfying both extras simultaneously. Create `rows`.

In [ ]:
def merge(left, left_names, right, right_names):
    matches = np.asarray(left_names)[:, None] == np.asarray(right_names)[None, :]
    right_indices = matches.argmax(axis=1)
    return np.concatenate((left, right[:, right_indices]), axis=0)

rows = merge(a_rows, names_a, b_rows, names_b)

In [ ]:
# Check your work
assert rows.shape == (1628, 7) and rows.dtype == np.float32
assert np.array_equal(rows[:len(a_rows)], a_rows)
for j, field in enumerate(names_a):
    assert np.array_equal(rows[len(a_rows):, j], b_rows[:, list(names_b).index(field)])
order = np.array([3, 0, 5, 6, 1, 4, 2])
assert np.array_equal(merge(a_rows, names_a, b_rows[:, order], np.asarray(names_b)[order]), rows)
print("Check passed.")

## C · Derive acceptance · 3 points

A panel passes if its mean rating is **at least 5** and every rating is **strictly greater than 3**. Write `accepted(scores)` to return a 1-D boolean mask. Create `fail_x` and `pass_x` containing only measurements, in original order. These inequalities deliberately differ from the released exam.

In [ ]:
def accepted(scores):
    return (scores.mean(axis=1) >= 5) & (scores.min(axis=1) > 3)

mask = accepted(rows[:, -3:])
fail_x, pass_x = rows[~mask, :-3], rows[mask, :-3]

In [ ]:
# Check your work
assert np.array_equal(accepted(np.array([[4, 5, 6], [3, 6, 8], [4, 5, 5]])), [True, False, False])
ref = (rows[:, -3:].sum(axis=1) >= 15) & (rows[:, -3:] > 3).all(axis=1)
assert np.array_equal(fail_x, rows[~ref, :-3]) and np.array_equal(pass_x, rows[ref, :-3])
print("Check passed.")

## D · Split within each class · 3 points

Independently permute the rows of each class. Use cuts at floor(0.70 n) and floor(0.85 n) within each class. Return three arrays per class as `fail_parts` and `pass_parts` in train, validation, test order. Print all six shapes.

In [ ]:
rng = np.random.default_rng(17)
def split_class(x):
    shuffled = x[rng.permutation(len(x))]
    return np.split(shuffled, [int(0.70*len(x)), int(0.85*len(x))])

fail_parts, pass_parts = split_class(fail_x), split_class(pass_x)
for label, parts in enumerate((fail_parts, pass_parts)):
    print(label, [part.shape for part in parts])

In [ ]:
# Check your work
for original, parts in ((fail_x, fail_parts), (pass_x, pass_parts)):
    n = len(original)
    assert list(map(len, parts)) == list(np.diff([0, int(.70*n), int(.85*n), n]))
    combined = np.concatenate(parts)
    assert not np.array_equal(combined, original)
    assert np.array_equal(np.unique(combined, axis=0, return_counts=True)[1], np.unique(original, axis=0, return_counts=True)[1])
    assert np.array_equal(np.unique(combined, axis=0), np.unique(original, axis=0))
print("Check passed.")

## E · Build reusable mixed batches · 3 points

Build `train_loader`, `val_loader`, `test_loader` with batch size 32. Each yields raw float32 measurements (B, 4) and float32 targets (B, 1), with 1 for acceptance. Mix classes in all three splits, preserve each sample once per pass, and keep the last partial batch.

In [ ]:
def mixed_loader(fails, passes):
    x = torch.from_numpy(np.concatenate((fails, passes)))
    y = torch.cat((torch.zeros(len(fails), 1), torch.ones(len(passes), 1)))
    return DataLoader(TensorDataset(x, y), batch_size=32, shuffle=True)

train_loader, val_loader, test_loader = [mixed_loader(f, p) for f, p in zip(fail_parts, pass_parts)]

In [ ]:
# Check your work
for loader, f, p in zip((train_loader, val_loader, test_loader), fail_parts, pass_parts):
    batches = list(loader)
    xs, ys = (torch.cat(t) for t in zip(*batches))
    assert xs.dtype == ys.dtype == torch.float32 and ys.shape == (len(f)+len(p), 1)
    assert xs.shape == (len(ys), 4) and torch.all((ys == 0) | (ys == 1))
    assert len(batches[0][0]) == 32 and any(y.unique().numel() == 2 for _, y in batches)
    for label, original in ((0, f), (1, p)):
        assert np.array_equal(np.unique(xs[ys[:, 0] == label].numpy(), axis=0), np.unique(original, axis=0))
print("Check passed.")

## F · Define the MLP · 2 points

Design a binary MLP with at least one nonlinear hidden layer and a single output logit. Define `Net(n_inputs)` as an `nn.Module`, infer `n_inputs` from a training batch and instantiate `model`. Verify its output on that batch.

In [ ]:
class Net(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(n_inputs, 24), nn.ReLU(),
                                    nn.Linear(24, 1))

    def forward(self, x):
        return self.layers(x)

n_inputs = next(iter(train_loader))[0].shape[1]
model = Net(n_inputs)

In [ ]:
# Check your work
xb, _ = next(iter(train_loader))
assert model(xb).shape == (len(xb), 1)
print("Check passed.")

## G · Train, restore and evaluate · 6 points

Train the model on `train_loader` with Adam and `nn.BCEWithLogitsLoss()`. Record one sample-weighted mean loss per epoch for each of training and validation. Use a maximum of 100 epochs and stop after eight consecutive epochs without a strictly lower validation loss. Save an independent snapshot on improvement and automatically restore the minimum-validation-loss weights, including when the epoch cap is reached.

Plot both loss histories together with a legend. In evaluation mode, gather aligned inputs, labels and logits for **all** test batches in a single pass. Report mean test loss and accuracy, then print five predicted/true label pairs. Keep the original input values for inspection. Use `y_test`, `x_test` and `logits_test` for your collected tensors, and `pred` for decisions with the same shape as `y_test`. Use `train_loss`, `val_loss`, `best_epoch` (zero-based), `test_loss` and `test_acc` for the other results.

In [ ]:
torch.manual_seed(41)
model = Net(n_inputs)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.006)
train_loss, val_loss = [], []
best_loss = float("inf")
stale = 0

def epoch_loss(loader, training):
    model.train(training)
    total, count = 0.0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            if training:
                optimizer.zero_grad()
            loss = criterion(model(x), y)
            if training:
                loss.backward()
                optimizer.step()
            total += loss.item() * len(y)
            count += len(y)
    return total / count

for epoch in range(100):
    train_loss.append(epoch_loss(train_loader, True))
    val_loss.append(epoch_loss(val_loader, False))
    if val_loss[-1] < best_loss:
        best_loss = val_loss[-1]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        stale = 0
    else:
        stale += 1
        if stale == 8:
            break
model.load_state_dict(best_state)
print("Restored epoch", best_epoch + 1)

fig, ax = plt.subplots()
ax.plot(train_loss, label="Training")
ax.plot(val_loss, label="Validation")
ax.set(xlabel="Epoch index", ylabel="Mean loss")
ax.legend()
plt.show()

model.eval()
inputs, targets, outputs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        inputs.append(x)
        targets.append(y)
        outputs.append(model(x))
x_test = torch.cat(inputs)
y_test = torch.cat(targets)
logits_test = torch.cat(outputs)
test_loss = criterion(logits_test, y_test).item()
pred = (logits_test >= 0).float()
test_acc = (pred == y_test).float().mean().item()
print(f"Test loss {test_loss:.3f}, accuracy {test_acc:.1%}")
print("Predicted | True")
print(torch.stack((pred.flatten()[:5], y_test.flatten()[:5]), dim=1).int())

In [ ]:
# Check your work
assert len(train_loss) == len(val_loss) and 1 <= len(val_loss) <= 100
assert np.isfinite(train_loss).all() and np.isfinite(val_loss).all()
assert best_epoch == np.argmin(val_loss)
assert len(val_loss) == 100 or len(val_loss) - 1 - best_epoch == 8
assert len(y_test) == len(test_loader.dataset) == len(x_test)
assert pred.shape == y_test.shape and not model.training
with torch.no_grad():
    assert torch.allclose(model(x_test), logits_test, atol=1e-5)
    ref_criterion = nn.BCEWithLogitsLoss()
    ref_val = sum(ref_criterion(model(x), y).item() * len(y) for x, y in val_loader)
assert np.isclose(ref_val / len(val_loader.dataset), min(val_loss), atol=1e-5)
assert np.isclose(test_loss, ref_criterion(logits_test, y_test).item(), atol=1e-5)
assert torch.equal(pred, (logits_test >= 0).float())
assert np.isclose(test_acc, (pred == y_test).float().mean().item())
print("Check passed.")

## H · Find the least confident panels · 3 points

Create a 1-D tensor `prob` of acceptance probabilities. In one vectorized ranking over the entire test set obtain exactly five distinct indices `least` closest to 0.5, from least to most confident. Print all four measurements, probability and true label for each.

In [ ]:
prob = logits_test.sigmoid().flatten()
least = (prob - 0.5).abs().argsort()[:5]
for i in least:
    print(x_test[i].tolist(), f"p={prob[i].item():.4f}", "true", int(y_test[i].item()))

In [ ]:
# Check your work
assert prob.shape == (len(y_test),) and torch.allclose(prob, logits_test.flatten().sigmoid())
assert len(least) == len(torch.unique(least)) == 5
assert torch.allclose((prob[least] - .5).abs(), (prob - .5).abs().sort().values[:5])
print("Check passed.")

## I · Show probabilities by true class · 3 points

Draw two histograms on a common probability scale, grouped by true class. Use the same bin edges from 0 to 1, sample counts on the vertical axis and a legend or clearly labeled panels. Check that the two histogram totals cover the test set.

In [ ]:
fig, ax = plt.subplots()
edges = np.linspace(0, 1, 16)
total = 0
for cls, label in enumerate(("Failed", "Accepted")):
    values = prob[y_test.flatten() == cls].numpy()
    counts, _, _ = ax.hist(values, bins=edges, alpha=0.55, label=label)
    total += int(counts.sum())
ax.set(xlabel="Predicted probability of acceptance", ylabel="Number of test panels", xlim=(0, 1))
ax.legend()
assert total == len(y_test)
plt.show()